## 1. Environment Setup & Configuration

Imports required libraries (PyTorch, RDKit, Hugging Face `datasets`, scikit-learn), runs a CUDA pre-flight check with a smoke test, sets random seeds for reproducibility, and initializes project directories (`data/`, `models/`, `metrics/`).

In [3]:
import argparse
import math
import os
import random
import re
import time
import urllib.request
import sys


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import Dataset as HFDataset, load_dataset
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, QED, Crippen, RDConfig
from rdkit.Contrib.SA_Score import sascorer
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from collections import Counter
from pathlib import Path

# Silence verbose RDKit parse warnings during validity checks
RDLogger.DisableLog("rdApp.*")

BATCH_SIZE = 512

SEED = 42


def resolve_allow_cpu() -> bool:
    """Check --allow-cpu CLI flag or ALLOW_CPU=1 environment variable (defaults to False)."""
    parser = argparse.ArgumentParser(add_help=False)
    parser.add_argument("--allow-cpu", action="store_true", default=False)
    args, _ = parser.parse_known_args(sys.argv[1:])
    env_flag = os.environ.get("ALLOW_CPU", "0").strip().lower() in {"1", "true", "yes"}
    return bool(args.allow_cpu or env_flag)


def validate_cuda_environment(allow_cpu: bool = False) -> torch.device:
    """Fail-fast CUDA pre-flight check with smoke test before loading datasets or models."""
    torch_ver = torch.__version__
    cuda_ver = torch.version.cuda
    cuda_avail = torch.cuda.is_available()

    print("=== Compute Environment Pre-Flight Check ===")
    print(f"Python executable  : {sys.executable}")
    print(f"PyTorch version    : {torch_ver}")
    print(f"CUDA build version : {cuda_ver}")
    print(f"CUDA available     : {cuda_avail}")

    if not cuda_avail:
        print("GPU name           : None")
        print("Total VRAM         : N/A")
        is_cpu_build = ("+cpu" in torch_ver) or (cuda_ver is None)
        if is_cpu_build:
            disk_ver = None
            try:
                ver_file = Path(torch.__file__).with_name("version.py")
                if ver_file.exists():
                    ns: dict = {}
                    exec(ver_file.read_text(encoding="utf-8"), ns)
                    disk_ver = ns.get("__version__")
            except Exception:
                pass

            if disk_ver and disk_ver != torch_ver and "+cu" in str(disk_ver):
                cause = (
                    f"Stale Jupyter kernel detected: in-memory PyTorch is '{torch_ver}', "
                    f"but .venv on disk already has CUDA build '{disk_ver}'. "
                    f"Restart the Jupyter kernel (Kernel -> Restart Kernel) and re-run from Cell 1."
                )
            else:
                cause = (
                    f"PyTorch is a CPU-only build (torch.__version__='{torch_ver}', "
                    f"torch.version.cuda={cuda_ver}). Reinstall a CUDA-enabled PyTorch wheel "
                    f"in .venv (e.g., `uv sync`) and restart the Jupyter kernel so training runs on GPU."
                )
        else:
            cause = (
                f"PyTorch was compiled with CUDA {cuda_ver}, but torch.cuda.is_available() "
                f"returned False. Check NVIDIA driver health (`nvidia-smi`), kernel module "
                f"(`nvidia_uvm`), device permissions (`/dev/nvidia*`), or CUDA_VISIBLE_DEVICES."
            )

        if not allow_cpu:
            raise RuntimeError(
                f"CUDA pre-flight check failed — refusing to silently fall back to CPU.\n"
                f"Reason: {cause}\n"
                f"Set ALLOW_CPU=1 or pass --allow-cpu only if you intentionally want a CPU debug run."
            )

        print(f"[WARNING] {cause}")
        print("[WARNING] Proceeding on CPU ONLY because ALLOW_CPU / --allow-cpu was explicitly enabled.")
        torch.set_num_threads(min(12, os.cpu_count() or 4))
        return torch.device("cpu")

    dev_idx = torch.cuda.current_device()
    device = torch.device("cuda", dev_idx)
    props = torch.cuda.get_device_properties(device)
    gpu_name = torch.cuda.get_device_name(device)
    total_gib = props.total_memory / (1024 ** 3)
    total_gb = props.total_memory / 1e9

    print(f"GPU name           : {gpu_name}")
    print(f"Total VRAM         : {total_gib:.2f} GiB ({total_gb:.2f} GB)")

    # Quick CUDA smoke test to catch driver / UVM / kernel launch issues immediately
    try:
        smoke_a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=device, dtype=torch.float32)
        smoke_out = (smoke_a @ smoke_a).sum()
        torch.cuda.synchronize(device)
        if smoke_out.device != device or not bool(torch.isfinite(smoke_out).all()):
            raise RuntimeError(f"Unexpected smoke-test tensor state: val={smoke_out}, device={smoke_out.device}")
        smoke_val = float(smoke_out.item())
        del smoke_a, smoke_out
    except Exception as exc:
        if not allow_cpu:
            raise RuntimeError(
                f"CUDA smoke test failed on {gpu_name} ({device}): {exc}. "
                f"This typically indicates an NVIDIA driver / nvidia_uvm / CUDA runtime issue."
            ) from exc
        print(f"[WARNING] CUDA smoke test failed ({exc}); falling back to CPU due to ALLOW_CPU / --allow-cpu.")
        torch.set_num_threads(min(12, os.cpu_count() or 4))
        return torch.device("cpu")

    print(f"CUDA smoke test    : PASSED (2x2 matmul sum={smoke_val:.1f} on {device})")
    return device


ALLOW_CPU = resolve_allow_cpu()
DEVICE = validate_cuda_environment(allow_cpu=ALLOW_CPU)
PIN_MEMORY = DEVICE.type == "cuda"
USE_AMP = DEVICE.type == "cuda"


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "metrics"
for d in (DATA_DIR, MODELS_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Active device      : {DEVICE} (AMP={USE_AMP}, pin_memory={PIN_MEMORY})")
print(f"Project root       : {PROJECT_ROOT}")


/home/mingoll/Projects/Mine/Molecule_Generation/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Compute Environment Pre-Flight Check ===
Python executable  : /home/mingoll/Projects/Mine/Molecule_Generation/.venv/bin/python
PyTorch version    : 2.15.0+cu132
CUDA build version : 13.2
CUDA available     : True
GPU name           : NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM         : 5.69 GiB (6.10 GB)
CUDA smoke test    : PASSED (2x2 matmul sum=54.0 on cuda:0)
Active device      : cuda:0 (AMP=True, pin_memory=True)
Project root       : /home/mingoll/Projects/Mine/Molecule_Generation


## 2. Data Gathering

Downloads and loads the two raw datasets used in this pipeline:
- **ZINC250K** (`yairschiff/zinc250k` via Hugging Face Datasets) for base model pre-training.
- **Tox21** (`tox21.csv.gz` from DeepChem S3) for non-toxic domain fine-tuning.

In [4]:
ds = load_dataset("yairschiff/zinc250k")

tox_url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
tox21_raw_df = pd.read_csv(tox_url, compression="gzip")

print(ds)
print(f"Tox21 shape: {tox21_raw_df.shape}")

DatasetDict({
    train: Dataset({
        features: ['smiles', 'logP', 'qed', 'SAS', 'canonical_smiles', 'single_bond', 'double_bond', 'triple_bond', 'aromatic_bond', 'ring_count', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R12', 'R13', 'R14', 'R15', 'R18', 'R24'],
        num_rows: 224568
    })
    validation: Dataset({
        features: ['smiles', 'logP', 'qed', 'SAS', 'canonical_smiles', 'single_bond', 'double_bond', 'triple_bond', 'aromatic_bond', 'ring_count', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R12', 'R13', 'R14', 'R15', 'R18', 'R24'],
        num_rows: 24887
    })
})
Tox21 shape: (7831, 14)


## 3. Data Preprocessing (ZINC250K)

Prepares the **ZINC250K** dataset for fingerprint-to-SMILES training:
- Subsets 50,000 training molecules and splits 4,000 validation molecules 50/50 into validation (2,000) and held-out test (2,000) sets.
- Builds an atom-wise SMILES tokenizer and vocabulary (52 tokens) from the training split.
- Converts each molecule into a 2048-bit Morgan fingerprint (radius 2) paired with padded token IDs (`FPToSMILESDataset`) and wraps them in PyTorch `DataLoader`s.

In [5]:
FP_SIZE = 2048
MFP_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_SIZE)

# On GPU use full ZINC250K; on CPU select a 50k train / 4k val subset so training stays fast
ZINC_TRAIN_SUBSET = 50_000
ZINC_VAL_SUBSET = 4_000

train_ds = ds["train"]
val_ds = ds["validation"]

train_ds = train_ds.shuffle(seed=SEED).select(range(min(ZINC_TRAIN_SUBSET, len(train_ds))))
val_ds = val_ds.shuffle(seed=SEED).select(range(min(ZINC_VAL_SUBSET, len(val_ds))))

# Split val_ds 50/50 into validation and held-out test splits
val_test_split = val_ds.train_test_split(test_size=0.5, seed=SEED)
zinc_val_hf = val_test_split["train"]
zinc_test_hf = val_test_split["test"]

# Atom-wise SMILES tokenizer & vocabulary
SMILES_REGEX = re.compile(
    r"(\[[^\[\]]{1,12}\]|Br|Cl|@@|@|%[0-9]{2}|[0-9]|"
    r"[A-Za-z]|=|#|-|\+|\\|/|:|~|\.|\(|\))"
)

SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
MAX_LEN = 92

def tokenize_smiles(smiles: str) -> list[str]:
    return SMILES_REGEX.findall(smiles.strip())

# Build vocabulary from training split
tok_counter = Counter()
for s in train_ds["canonical_smiles"]:
    tok_counter.update(tokenize_smiles(s))

vocab_tokens = SPECIAL_TOKENS + sorted([t for t, c in tok_counter.items() if c >= 2])
token2idx = {t: i for i, t in enumerate(vocab_tokens)}
idx2token = {i: t for t, i in token2idx.items()}
print(f"Vocabulary size: {len(vocab_tokens)} tokens")


class FPToSMILESDataset(Dataset):
    def __init__(self, hf_dataset, token2idx: dict[str, int], smiles_col: str = "canonical_smiles", max_len: int = MAX_LEN):
        smiles_list = hf_dataset[smiles_col]
        n = len(smiles_list)

        # Store fingerprints as uint8 (4x less RAM than float32), cast to float32 per item
        fps = np.empty((n, FP_SIZE), dtype=np.uint8)
        tokens = np.full((n, max_len), PAD_IDX, dtype=np.int64)

        valid_count = 0
        for smi in smiles_list:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue  
            
            fps[valid_count] = MFP_GEN.GetFingerprintAsNumPy(mol)
            toks = tokenize_smiles(smi)[: max_len - 2]
            ids = [BOS_IDX] + [token2idx.get(t, UNK_IDX) for t in toks] + [EOS_IDX]
            tokens[valid_count, : len(ids)] = ids
            valid_count += 1

        self.fps = fps[:valid_count]
        self.tokens = tokens[:valid_count]

    def __len__(self) -> int:
        return len(self.fps)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.fps[idx]).float(), torch.from_numpy(self.tokens[idx])

PIN_MEM = DEVICE.type == "cuda"
zinc_train_loader = DataLoader(FPToSMILESDataset(train_ds, token2idx), batch_size=BATCH_SIZE, shuffle=True, pin_memory=PIN_MEM)
zinc_val_loader = DataLoader(FPToSMILESDataset(zinc_val_hf, token2idx), batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEM)
zinc_test_loader = DataLoader(FPToSMILESDataset(zinc_test_hf, token2idx), batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEM)

print(f"DataLoaders -> Train: {len(zinc_train_loader.dataset):,} | Val: {len(zinc_val_loader.dataset):,} | Test: {len(zinc_test_loader.dataset):,}")

Vocabulary size: 52 tokens
DataLoaders -> Train: 50,000 | Val: 2,000 | Test: 2,000


## 4. Training (ZINC250K Base Model)

Defines the `FingerprintToSMILES` causal Transformer decoder and trains `zinc_model` on **ZINC250K** (`zinc_train_loader`) for 12 epochs using AdamW, `OneCycleLR`, and mixed precision (AMP). Tracks validation loss/perplexity/token accuracy and quick generation metrics each epoch, restoring the best validation checkpoint.

In [6]:
class FingerprintToSMILES(nn.Module):
    def __init__(
        self,
        fp_size: int = FP_SIZE,
        vocab_size: int = len(vocab_tokens),
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 1024,
        max_len: int = MAX_LEN,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.config = {
            "fp_size": fp_size,
            "vocab_size": vocab_size,
            "d_model": d_model,
            "nhead": nhead,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "max_len": max_len,
            "dropout": dropout,
        }
        self.d_model = d_model
        self.max_len = max_len

        self.fp_proj = nn.Sequential(
            nn.Linear(fp_size, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        self.tok_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, fp: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = input_ids.shape
        if seq_len > self.max_len:
            raise ValueError(f"Sequence length {seq_len} exceeds max_len={self.max_len}")

        fp_prefix = self.fp_proj(fp).unsqueeze(1)

        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        tok_emb = self.tok_embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(positions)
        tok_emb = self.emb_dropout(tok_emb)

        x = torch.cat([fp_prefix, tok_emb], dim=1)
        total_len = seq_len + 1

        causal_mask = torch.triu(
            torch.ones((total_len, total_len), device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        key_padding_mask = torch.cat(
            [
                torch.zeros((bsz, 1), device=x.device, dtype=torch.bool),
                input_ids == PAD_IDX,
            ],
            dim=1,
        )

        out = self.transformer(x, mask=causal_mask, src_key_padding_mask=key_padding_mask)
        out = self.norm(out)

        return self.fc(out[:, 1:, :])

    @torch.no_grad()
    def generate(
        self,
        fp: torch.Tensor,
        idx2token: dict[int, str],
        max_len: int = MAX_LEN,
        temperature: float = 0.85,
        top_p: float = 0.9,
    ) -> list[str]:
        self.eval()
        bsz = fp.size(0)
        cur_ids = torch.full((bsz, 1), BOS_IDX, dtype=torch.long, device=fp.device)
        finished = torch.zeros(bsz, dtype=torch.bool, device=fp.device)
        decoded = [[] for _ in range(bsz)]

        for _ in range(max_len - 1):
            logits = self.forward(fp, cur_ids)[:, -1, :]
            logits[:, [PAD_IDX, BOS_IDX, UNK_IDX]] = float("-inf")

            if temperature <= 0.0:
                next_tok = logits.argmax(dim=-1, keepdim=True)
            else:
                scaled_logits = logits / temperature
                if 0.0 < top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True, dim=-1)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                    sorted_mask = cumulative_probs > top_p
                    sorted_mask[:, 1:] = sorted_mask[:, :-1].clone()
                    sorted_mask[:, 0] = False

                    sorted_logits = sorted_logits.masked_fill(sorted_mask, float("-inf"))
                    filtered_logits = torch.full_like(scaled_logits, float("-inf"))
                    filtered_logits.scatter_(1, sorted_indices, sorted_logits)
                else:
                    filtered_logits = scaled_logits

                probs = F.softmax(filtered_logits, dim=-1)
                next_tok = torch.multinomial(probs, num_samples=1)

            for i in range(bsz):
                if not finished[i]:
                    tid = int(next_tok[i, 0].item())
                    if tid == EOS_IDX:
                        finished[i] = True
                    else:
                        decoded[i].append(idx2token.get(tid, ""))

            if bool(finished.all()):
                break
            cur_ids = torch.cat([cur_ids, next_tok], dim=1)

        return ["".join(seq) for seq in decoded]


def trim_tokens(tokens: torch.Tensor) -> torch.Tensor:
    max_active = max(int((tokens != PAD_IDX).sum(dim=1).max().item()), 2)
    return tokens[:, :max_active]


@torch.no_grad()
def evaluate_val(model: nn.Module, loader: DataLoader, criterion: nn.Module, use_amp: bool = False) -> dict:
    model.eval()
    total_loss, total_correct, total_toks = 0.0, 0, 0
    for fp, tokens in loader:
        tokens = trim_tokens(tokens).to(DEVICE, non_blocking=True)
        fp = fp.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits = model(fp, tokens[:, :-1])
            targets = tokens[:, 1:]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        mask = targets != PAD_IDX
        n_tok = int(mask.sum().item())
        preds = logits.argmax(dim=-1)
        total_correct += int((preds[mask] == targets[mask]).sum().item())
        total_loss += float(loss.item()) * n_tok
        total_toks += n_tok

    avg_loss = total_loss / max(total_toks, 1)
    return {
        "loss": avg_loss,
        "ppl": math.exp(min(avg_loss, 20.0)),
        "tok_acc": total_correct / max(total_toks, 1),
    }


@torch.no_grad()
def quick_gen_metrics(
    model: nn.Module,
    loader: DataLoader,
    idx2token: dict[int, str],
    n_samples: int = 128,
    temperature: float = 0.0,
) -> dict:
    model.eval()
    fp_batch, _ = next(iter(loader))
    fp_sample = fp_batch[:n_samples].to(DEVICE, non_blocking=True)
    true_fps = (fp_sample.cpu().numpy() > 0.5)

    gen_smiles = model.generate(fp_sample, idx2token, temperature=temperature)

    valid_canon = []
    precs, recs, f1s = [], [], []

    for smi, true_fp in zip(gen_smiles, true_fps):
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol is None:
            continue
        valid_canon.append(Chem.MolToSmiles(mol, canonical=True))

        gen_fp = MFP_GEN.GetFingerprintAsNumPy(mol).astype(bool)
        tp = int(np.logical_and(gen_fp, true_fp).sum())
        pred_pos = int(gen_fp.sum())
        true_pos = int(true_fp.sum())

        p = tp / pred_pos if pred_pos > 0 else 0.0
        r = tp / true_pos if true_pos > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precs.append(p)
        recs.append(r)
        f1s.append(f1)

    n_valid = len(valid_canon)
    return {
        "validity": n_valid / max(len(gen_smiles), 1),
        "uniqueness": len(set(valid_canon)) / max(n_valid, 1) if n_valid > 0 else 0.0,
        "fp_prec": float(np.mean(precs)) if precs else 0.0,
        "fp_rec": float(np.mean(recs)) if recs else 0.0,
        "fp_f1": float(np.mean(f1s)) if f1s else 0.0,
        "sample": valid_canon[0] if valid_canon else (gen_smiles[0] if gen_smiles else "<empty>"),
    }


USE_AMP = DEVICE.type == "cuda"
model_kwargs = {"d_model": 256, "nhead": 8, "num_layers": 4, "dim_feedforward": 1024}
zinc_model = FingerprintToSMILES(**model_kwargs).to(DEVICE)
model_param_device = next(zinc_model.parameters()).device
assert model_param_device == DEVICE, (
    f"zinc_model parameters are on {model_param_device}, expected target device {DEVICE}"
)
print(f"Model parameters: {sum(p.numel() for p in zinc_model.parameters()):,} | Device: {model_param_device} | AMP: {USE_AMP}")

ZINC_EPOCHS = 12
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(zinc_model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    epochs=ZINC_EPOCHS,
    steps_per_epoch=len(zinc_train_loader),
    pct_start=0.1,
)
scaler = torch.amp.GradScaler("cuda" if USE_AMP else "cpu", enabled=USE_AMP)

best_val_loss = float("inf")
best_zinc_weights = None

for epoch in range(1, ZINC_EPOCHS + 1):
    t0 = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    zinc_model.train()
    train_loss_sum, train_correct, train_toks = 0.0, 0, 0

    for fp, tokens in zinc_train_loader:
        tokens = trim_tokens(tokens).to(DEVICE, non_blocking=True)
        fp = fp.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = zinc_model(fp, tokens[:, :-1])
            targets = tokens[:, 1:]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(zinc_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        with torch.no_grad():
            mask = targets != PAD_IDX
            n_tok = int(mask.sum().item())
            train_correct += int((logits.argmax(dim=-1)[mask] == targets[mask]).sum().item())
            train_loss_sum += float(loss.item()) * n_tok
            train_toks += n_tok

    train_loss = train_loss_sum / max(train_toks, 1)
    train_acc = train_correct / max(train_toks, 1)
    lr = optimizer.param_groups[0]["lr"]

    val = evaluate_val(zinc_model, zinc_val_loader, criterion, use_amp=USE_AMP)
    gen = quick_gen_metrics(zinc_model, zinc_val_loader, idx2token, n_samples=128, temperature=0.0)

    is_best = val["loss"] < best_val_loss
    if is_best:
        best_val_loss = val["loss"]
        best_zinc_weights = {k: v.detach().cpu().clone() for k, v in zinc_model.state_dict().items()}

    if DEVICE.type == "cuda":
        peak_vram_bytes = torch.cuda.max_memory_allocated(DEVICE)
        total_vram_bytes = torch.cuda.get_device_properties(DEVICE).total_memory
        vram_str = (
            f" | Peak VRAM: {peak_vram_bytes / (1024 ** 3):.2f}/{total_vram_bytes / (1024 ** 3):.2f} GiB "
            f"({peak_vram_bytes / (1024 ** 2):.0f} MiB)"
        )
    else:
        vram_str = ""
    best_tag = " [BEST]" if is_best else ""

    print(
        f"[ZINC250K] Epoch {epoch:02d}/{ZINC_EPOCHS:02d}{best_tag} | "
        f"LR: {lr:.2e} | Time: {time.time() - t0:.1f}s{vram_str}\n"
        f"  Train -> Loss: {train_loss:.4f} | PPL: {math.exp(min(train_loss, 20.0)):.2f} | TokAcc: {train_acc:.4f}\n"
        f"  Val   -> Loss: {val['loss']:.4f} | PPL: {val['ppl']:.2f} | TokAcc: {val['tok_acc']:.4f}\n"
        f"  Gen   -> Valid: {gen['validity']:.4f} | Unique: {gen['uniqueness']:.4f} | "
        f"FP Prec: {gen['fp_prec']:.4f} | Rec: {gen['fp_rec']:.4f} | F1: {gen['fp_f1']:.4f}\n"
        f"  Sample: {gen['sample']}"
    )

zinc_model.load_state_dict(best_zinc_weights)

Model parameters: 3,734,836 | Device: cuda:0 | AMP: True
[ZINC250K] Epoch 01/12 [BEST] | LR: 9.41e-04 | Time: 17.6s | Peak VRAM: 1.92/5.69 GiB (1966 MiB)
  Train -> Loss: 1.9003 | PPL: 6.69 | TokAcc: 0.4452
  Val   -> Loss: 1.2143 | PPL: 3.37 | TokAcc: 0.5983
  Gen   -> Valid: 0.0859 | Unique: 1.0000 | FP Prec: 0.3359 | Rec: 0.2689 | F1: 0.2974
  Sample: COc1cccc(NC(=O)N2CCCCCCCCCCCC2)c1
[ZINC250K] Epoch 02/12 [BEST] | LR: 9.86e-04 | Time: 15.5s | Peak VRAM: 1.92/5.69 GiB (1966 MiB)
  Train -> Loss: 1.0157 | PPL: 2.76 | TokAcc: 0.6569
  Val   -> Loss: 0.7478 | PPL: 2.11 | TokAcc: 0.7380
  Gen   -> Valid: 0.1719 | Unique: 1.0000 | FP Prec: 0.4219 | Rec: 0.3697 | F1: 0.3898
  Sample: CC(C)[NH+]1CCCN(C(=O)[O-])C1
[ZINC250K] Epoch 03/12 [BEST] | LR: 9.32e-04 | Time: 15.5s | Peak VRAM: 1.92/5.69 GiB (1966 MiB)
  Train -> Loss: 0.7401 | PPL: 2.10 | TokAcc: 0.7387
  Val   -> Loss: 0.5936 | PPL: 1.81 | TokAcc: 0.7870
  Gen   -> Valid: 0.2656 | Unique: 1.0000 | FP Prec: 0.5228 | Rec: 0.4432 | F

<All keys matched successfully>

## 5. Evaluation Metrics Setup (Drug-Likeness & Synthesizability)

Loads RDKit's `SA_Score` module and extends `quick_gen_metrics` to evaluate generated molecules on **validity**, **uniqueness**, **fingerprint Precision / Recall / F1**, and drug-likeness properties (**logP**, **QED**, **SAS**, and overall **suitability**).

In [7]:
# Load RDKit's built-in SA_Score module
sa_dir = os.path.join(RDConfig.RDContribDir, "SA_Score")
print(sa_dir)
if sa_dir not in sys.path:
    sys.path.append(sa_dir)

@torch.no_grad()
def quick_gen_metrics(
    model: nn.Module,
    loader: DataLoader,
    idx2token: dict[int, str],
    n_samples: int = 128,
    temperature: float = 0.0,
) -> dict:
    model.eval()
    fp_batch, _ = next(iter(loader))
    fp_sample = fp_batch[:n_samples].to(DEVICE, non_blocking=True)
    true_fps = (fp_sample.cpu().numpy() > 0.5)

    gen_smiles = model.generate(fp_sample, idx2token, temperature=temperature)

    valid_canon = []
    precs, recs, f1s = [], [], []
    logps, qeds, sass = [], [], []
    suitable = 0

    for smi, true_fp in zip(gen_smiles, true_fps):
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol is None:
            continue
        valid_canon.append(Chem.MolToSmiles(mol, canonical=True))

        # Fingerprint match
        gen_fp = MFP_GEN.GetFingerprintAsNumPy(mol).astype(bool)
        tp = int(np.logical_and(gen_fp, true_fp).sum())
        pred_pos = int(gen_fp.sum())
        true_pos = int(true_fp.sum())

        p = tp / pred_pos if pred_pos > 0 else 0.0
        r = tp / true_pos if true_pos > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precs.append(p)
        recs.append(r)
        f1s.append(f1)

        # Drug-likeness & synthesizability
        logp = float(Crippen.MolLogP(mol))
        qed = float(QED.qed(mol))
        sas = float(sascorer.calculateScore(mol))
        logps.append(logp)
        qeds.append(qed)
        sass.append(sas)

        if (-0.4 <= logp <= 5.0) and (qed >= 0.5) and (sas <= 4.5):
            suitable += 1

    n_valid = len(valid_canon)
    return {
        "validity": n_valid / max(len(gen_smiles), 1),
        "uniqueness": len(set(valid_canon)) / max(n_valid, 1) if n_valid > 0 else 0.0,
        "fp_prec": float(np.mean(precs)) if precs else 0.0,
        "fp_rec": float(np.mean(recs)) if recs else 0.0,
        "fp_f1": float(np.mean(f1s)) if f1s else 0.0,
        "logP": float(np.mean(logps)) if logps else 0.0,
        "QED": float(np.mean(qeds)) if qeds else 0.0,
        "SAS": float(np.mean(sass)) if sass else 0.0,
        "suitable": suitable / max(n_valid, 1) if n_valid > 0 else 0.0,
        "sample": valid_canon[0] if valid_canon else (gen_smiles[0] if gen_smiles else "<empty>"),
    }

/home/mingoll/Projects/Mine/Molecule_Generation/.venv/lib/python3.15/site-packages/rdkit/Contrib/SA_Score


## 6. Evaluating & Saving Base Model (`zinc_model` on ZINC250K Validation Data)

Evaluates the trained base `zinc_model` on `zinc_val_loader` (loss, perplexity, token accuracy, and 500-sample generation metrics), saves the checkpoint (`models/zinc250k_fp2smiles_v1.pt`) with vocabulary and metadata, and verifies loading with a round-trip sanity check.

In [8]:
ZINC_MODEL_PATH = MODELS_DIR / "zinc250k_fp2smiles_v1.pt"

def save_model_file(path: Path, model: FingerprintToSMILES, version_name: str, metrics: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "version": version_name,
            "model_config": model.config,
            "token2idx": token2idx,
            "idx2token": idx2token,
            "metrics": metrics,
            "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        },
        path,
    )

def load_model_file(path: Path, device: torch.device = DEVICE) -> tuple[FingerprintToSMILES, dict]:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = FingerprintToSMILES(**ckpt["model_config"]).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    return m, ckpt

zinc_eval_metrics = {
    **evaluate_val(zinc_model, zinc_val_loader, criterion, use_amp=USE_AMP),
    **quick_gen_metrics(zinc_model, zinc_val_loader, idx2token, n_samples=500, temperature=0.0),
}
save_model_file(ZINC_MODEL_PATH, zinc_model, "v1.0-zinc250k-fp2smiles", zinc_eval_metrics)
print(f"Saved model -> {ZINC_MODEL_PATH} ({ZINC_MODEL_PATH.stat().st_size / (1024 * 1024):.2f} MB)")

# Quick round-trip sanity check
_test_model, _test_ckpt = load_model_file(ZINC_MODEL_PATH)
assert _test_ckpt["version"] == "v1.0-zinc250k-fp2smiles"
del _test_model, _test_ckpt
print("Load check passed.")

Saved model -> /home/mingoll/Projects/Mine/Molecule_Generation/models/zinc250k_fp2smiles_v1.pt (14.27 MB)
Load check passed.


## 7. Data Preprocessing (Tox21 Non-Toxic Subset)

Preprocesses the **Tox21** dataset to isolate clean, non-toxic molecules for fine-tuning:
- Strips salts (keeping the largest fragment) and keeps only molecules with zero positive toxicity outcomes across all 12 Tox21 assays.
- Canonicalizes SMILES, removes duplicates, and filters to sequences compatible with the ZINC250K vocabulary (4,697 molecules).
- Splits 80/10/10 into `tox_train_loader`, `tox_val_loader`, and `tox_test_loader`.

In [9]:
TOX21_ASSAYS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER", "NR-ER-LBD",
    "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

non_toxic_smiles = []
seen_tox = set()

for _, row in tox21_raw_df.iterrows():
    if pd.isna(row["smiles"]):
        continue
    smi = str(row["smiles"]).strip()
    if not smi:
        continue
    # Strip salts by keeping largest fragment
    if "." in smi:
        smi = max(smi.split("."), key=len)

    assay_vals = row[TOX21_ASSAYS].dropna().values.astype(int)
    # Keep molecules with 0 positive toxicity outcomes across tested assays
    if len(assay_vals) == 0 or (assay_vals == 1).sum() > 0:
        continue

    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() < 3:
        continue
    can_smi = Chem.MolToSmiles(mol, canonical=True)
    toks = tokenize_smiles(can_smi)

    if (
        can_smi not in seen_tox
        and 4 <= len(toks) <= MAX_LEN - 2
        and all(t in token2idx for t in toks)
    ):
        seen_tox.add(can_smi)
        non_toxic_smiles.append(can_smi)

print(f"Clean non-toxic Tox21 molecules compatible with vocab: {len(non_toxic_smiles):,}")

tox_hf = HFDataset.from_dict({"canonical_smiles": non_toxic_smiles})

tox_split_1 = tox_hf.train_test_split(test_size=0.20, seed=SEED)
tox_split_2 = tox_split_1["test"].train_test_split(test_size=0.50, seed=SEED)

tox_train_hf = tox_split_1["train"]
tox_val_hf = tox_split_2["train"]
tox_test_hf = tox_split_2["test"]

PIN_MEM = DEVICE.type == "cuda"
tox_train_loader = DataLoader(FPToSMILESDataset(tox_train_hf, token2idx), batch_size=64, shuffle=True, pin_memory=PIN_MEM)
tox_val_loader = DataLoader(FPToSMILESDataset(tox_val_hf, token2idx), batch_size=64, shuffle=False, pin_memory=PIN_MEM)
tox_test_loader = DataLoader(FPToSMILESDataset(tox_test_hf, token2idx), batch_size=64, shuffle=False, pin_memory=PIN_MEM)

print(
    f"Tox21 Non-Toxic Splits -> Train: {len(tox_train_loader.dataset):,} | "
    f"Val: {len(tox_val_loader.dataset):,} | Test: {len(tox_test_loader.dataset):,}"
)

Clean non-toxic Tox21 molecules compatible with vocab: 4,697
Tox21 Non-Toxic Splits -> Train: 3,757 | Val: 470 | Test: 470


## 8. Fine-Tuning (`zinc_model` on Tox21 Non-Toxic Data)

Loads the pre-trained ZINC250K checkpoint into `tox_model`, records its zero-shot baseline on `tox_val_loader`, and fine-tunes on `tox_train_loader` for 10 epochs using AdamW (`lr=2e-4`) and `CosineAnnealingLR`. Restores the best validation weights after training.

In [10]:
tox_model, tox_ckpt = load_model_file(ZINC_MODEL_PATH)
tox_idx2token = tox_ckpt["idx2token"]

USE_AMP = DEVICE.type == "cuda"
tox_param_device = next(tox_model.parameters()).device
assert tox_param_device == DEVICE, (
    f"tox_model parameters are on {tox_param_device}, expected target device {DEVICE}"
)
print(f"Tox21 model device: {tox_param_device} | AMP: {USE_AMP}")

FT_EPOCHS = 10
ft_optimizer = torch.optim.AdamW(tox_model.parameters(), lr=2e-4, weight_decay=1e-2)
ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=2e-5)
ft_scaler = torch.amp.GradScaler("cuda" if USE_AMP else "cpu", enabled=USE_AMP)

init_val = evaluate_val(tox_model, tox_val_loader, criterion, use_amp=USE_AMP)
best_tox_val = init_val["loss"]
best_tox_weights = {k: v.detach().cpu().clone() for k, v in tox_model.state_dict().items()}
print(
    f"Initial Tox21 Val (before fine-tuning) -> "
    f"Loss: {init_val['loss']:.4f} | PPL: {init_val['ppl']:.2f} | TokAcc: {init_val['tok_acc']:.4f}"
)

for epoch in range(1, FT_EPOCHS + 1):
    t0 = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    tox_model.train()
    train_loss_sum, train_correct, train_toks = 0.0, 0, 0

    for fp, tokens in tox_train_loader:
        tokens = trim_tokens(tokens).to(DEVICE, non_blocking=True)
        fp = fp.to(DEVICE, non_blocking=True)

        ft_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = tox_model(fp, tokens[:, :-1])
            targets = tokens[:, 1:]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        ft_scaler.scale(loss).backward()
        ft_scaler.unscale_(ft_optimizer)
        nn.utils.clip_grad_norm_(tox_model.parameters(), 1.0)
        ft_scaler.step(ft_optimizer)
        ft_scaler.update()

        with torch.no_grad():
            mask = targets != PAD_IDX
            n_tok = int(mask.sum().item())
            train_correct += int((logits.argmax(dim=-1)[mask] == targets[mask]).sum().item())
            train_loss_sum += float(loss.item()) * n_tok
            train_toks += n_tok

    ft_scheduler.step()

    train_loss = train_loss_sum / max(train_toks, 1)
    train_acc = train_correct / max(train_toks, 1)
    lr = ft_optimizer.param_groups[0]["lr"]

    val = evaluate_val(tox_model, tox_val_loader, criterion, use_amp=USE_AMP)
    gen = quick_gen_metrics(tox_model, tox_val_loader, tox_idx2token, n_samples=64, temperature=0.0)

    is_best = val["loss"] < best_tox_val
    if is_best:
        best_tox_val = val["loss"]
        best_tox_weights = {k: v.detach().cpu().clone() for k, v in tox_model.state_dict().items()}

    if DEVICE.type == "cuda":
        peak_vram_bytes = torch.cuda.max_memory_allocated(DEVICE)
        total_vram_bytes = torch.cuda.get_device_properties(DEVICE).total_memory
        vram_str = (
            f" | Peak VRAM: {peak_vram_bytes / (1024 ** 3):.2f}/{total_vram_bytes / (1024 ** 3):.2f} GiB "
            f"({peak_vram_bytes / (1024 ** 2):.0f} MiB)"
        )
    else:
        vram_str = ""
    best_tag = " [BEST]" if is_best else ""

    print(
        f"[Tox21 Fine-Tune] Epoch {epoch:02d}/{FT_EPOCHS:02d}{best_tag} | "
        f"LR: {lr:.2e} | Time: {time.time() - t0:.1f}s{vram_str}\n"
        f"  Train -> Loss: {train_loss:.4f} | PPL: {math.exp(min(train_loss, 20.0)):.2f} | TokAcc: {train_acc:.4f}\n"
        f"  Val   -> Loss: {val['loss']:.4f} | PPL: {val['ppl']:.2f} | TokAcc: {val['tok_acc']:.4f}\n"
        f"  Gen   -> Valid: {gen['validity']:.4f} | Unique: {gen['uniqueness']:.4f} | "
        f"FP Prec: {gen['fp_prec']:.4f} | Rec: {gen['fp_rec']:.4f} | F1: {gen['fp_f1']:.4f}\n"
        f"  Props -> logP: {gen['logP']:.2f} | QED: {gen['QED']:.3f} | SAS: {gen['SAS']:.2f} | "
        f"Suitable: {gen['suitable']:.2%}\n"
        f"  Sample: {gen['sample']}"
    )

tox_model.load_state_dict(best_tox_weights)

Tox21 model device: cuda:0 | AMP: True
Initial Tox21 Val (before fine-tuning) -> Loss: 1.0123 | PPL: 2.75 | TokAcc: 0.7442
[Tox21 Fine-Tune] Epoch 01/10 [BEST] | LR: 1.96e-04 | Time: 1.6s | Peak VRAM: 0.44/5.69 GiB (447 MiB)
  Train -> Loss: 0.7110 | PPL: 2.04 | TokAcc: 0.7618
  Val   -> Loss: 0.5786 | PPL: 1.78 | TokAcc: 0.8024
  Gen   -> Valid: 0.7969 | Unique: 1.0000 | FP Prec: 0.6692 | Rec: 0.6350 | F1: 0.6408
  Props -> logP: 1.85 | QED: 0.522 | SAS: 2.30 | Suitable: 54.90%
  Sample: CN(C)CCN(Cc1ccsc1)c1ccccc1
[Tox21 Fine-Tune] Epoch 02/10 [BEST] | LR: 1.83e-04 | Time: 2.3s | Peak VRAM: 0.44/5.69 GiB (447 MiB)
  Train -> Loss: 0.5727 | PPL: 1.77 | TokAcc: 0.7992
  Val   -> Loss: 0.5385 | PPL: 1.71 | TokAcc: 0.8151
  Gen   -> Valid: 0.6094 | Unique: 1.0000 | FP Prec: 0.7460 | Rec: 0.7047 | F1: 0.7121
  Props -> logP: 1.98 | QED: 0.514 | SAS: 2.37 | Suitable: 48.72%
  Sample: CN(C)CCN(Cc1ccsc1)c1ccccc1
[Tox21 Fine-Tune] Epoch 03/10 [BEST] | LR: 1.63e-04 | Time: 1.7s | Peak VRAM: 0.4

<All keys matched successfully>

## 9. Evaluating F1 & Generation Metrics (`tox_model` on Tox21 Test Data)

Evaluates the fine-tuned `tox_model` on the held-out **Tox21 test set** (`tox_test_loader`), reporting validity, uniqueness, fingerprint **Precision / Recall / F1**, drug-likeness (**logP**, **QED**, **SAS**, **suitability**), and a sample generated SMILES.

In [15]:
tox_eval_metrics = quick_gen_metrics(
    tox_model,
    tox_test_loader,
    tox_idx2token,
    n_samples=len(tox_test_loader.dataset),
    temperature=0.0,
)
print("=== Tox21 Fine-Tuned Test Evaluation ===")
for k, v in tox_eval_metrics.items():
    if k != "sample":
        print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")
print("Sample valid generated SMILES:", tox_eval_metrics["sample"])

=== Tox21 Fine-Tuned Test Evaluation ===
  validity            : 0.7812
  uniqueness          : 1.0000
  fp_prec             : 0.7202
  fp_rec              : 0.7013
  fp_f1               : 0.7046
  logP                : 2.1305
  QED                 : 0.5272
  SAS                 : 2.4875
  suitable            : 0.5800
Sample valid generated SMILES: CN(C)/C=C/c1ccc(Br)cc1
